In [20]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
import pdfplumber


CURRENT = Path.cwd()

if CURRENT.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT.parent
else:
    PROJECT_ROOT = CURRENT


PDF_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "daily"
)

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


# Use the exact filename from Notebook 02
PDF_NAME = "IDSP-Daily-Report-01.09.2026.pdf"

PDF_PATH = PDF_FOLDER / PDF_NAME


if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}"
    )


print("Using PDF:", PDF_PATH.name)

Using PDF: IDSP-Daily-Report-01.09.2026.pdf


In [21]:
def clean(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def optional_count(value):
    text = clean(value)

    text = text.replace(",", "")

    if text == "-":
        return 0

    if text == "":
        return pd.NA

    if text.isdigit():
        return int(text)

    raise ValueError(
        f"Invalid count: {value!r}"
    )

In [22]:
TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "intersection_tolerance": 5,
    "snap_tolerance": 3,
    "join_tolerance": 3,
}


with pdfplumber.open(PDF_PATH) as pdf:
    if len(pdf.pages) < 3:
        raise ValueError(
            "The PDF must contain at least three pages."
        )

    page_2_tables = pdf.pages[1].extract_tables(
        TABLE_SETTINGS
    )

    page_3_tables = pdf.pages[2].extract_tables(
        TABLE_SETTINGS
    )


if not page_2_tables:
    raise ValueError(
        "No table was found on page 2."
    )


if not page_3_tables:
    raise ValueError(
        "No table was found on page 3."
    )


page_2_table = max(
    page_2_tables,
    key=len
)

page_3_table = max(
    page_3_tables,
    key=len
)


print(
    "Page 2 rows:",
    len(page_2_table)
)

print(
    "Page 3 rows:",
    len(page_3_table)
)

Page 2 rows: 40
Page 3 rows: 32


In [23]:
header_text = " ".join(
    clean(cell)
    for row in page_2_table[:3]
    for cell in row
)


date_match = re.search(
    r"\b\d{2}-\d{2}-\d{2}\b",
    header_text
)


if date_match is None:
    raise ValueError(
        "Report date not found."
    )


report_date = datetime.strptime(
    date_match.group(),
    "%d-%m-%y"
)


report_date = (
    report_date
    .date()
    .isoformat()
)


print("Report date:", report_date)

Report date: 2026-09-01


In [24]:
DISTRICTS = {
    "TVM": "Thiruvananthapuram",
    "KLM": "Kollam",
    "PTA": "Pathanamthitta",
    "IDK": "Idukki",
    "KTM": "Kottayam",
    "ALP": "Alappuzha",
    "EKM": "Ernakulam",
    "TSR": "Thrissur",
    "PKD": "Palakkad",
    "MPM": "Malappuram",
    "KKD": "Kozhikode",
    "WYD": "Wayanad",
    "KNR": "Kannur",
    "KSD": "Kasaragod",
}


DISEASE_ALIASES = {
    "Lepto": "Leptospirosis",
    "CG": "Chikungunya",
    "Chicken Pox": "Chickenpox",
    "Hep A": "Hepatitis A",
    "H1N1": "Influenza A (H1N1)",
    "Amoebic Meningo Encephalitis":
        "Amoebic Meningoencephalitis",
}


def normalize_disease(disease_name):
    disease_name = clean(disease_name)

    return DISEASE_ALIASES.get(
        disease_name,
        disease_name
    )

In [25]:
district_code_pattern = "|".join(
    re.escape(code)
    for code in DISTRICTS
)


DISTRICT_MARKER = re.compile(
    rf"\b({district_code_pattern})\s*:"
)


def split_district_sections(text):
    text = clean(text)

    matches = list(
        DISTRICT_MARKER.finditer(text)
    )

    sections = []


    for index, match in enumerate(matches):
        district_code = match.group(1)

        locality_start = match.end()


        if index + 1 < len(matches):
            locality_end = (
                matches[index + 1].start()
            )
        else:
            locality_end = len(text)


        locality_text = text[
            locality_start:locality_end
        ]


        locality_text = locality_text.strip(
            " ,;"
        )


        if locality_text:
            sections.append(
                (
                    district_code,
                    locality_text
                )
            )


    return sections

In [26]:
locality_start = None


for index, row in enumerate(
    page_2_table
):
    first_cell = clean(row[0])

    if first_cell.upper() == "DISEASE":
        locality_start = index + 1
        break


if locality_start is None:
    raise ValueError(
        "The locality section was not found."
    )


print(
    "Locality section starts at row:",
    locality_start
)

Locality section starts at row: 22


In [27]:
locality_records = []

current_disease = None


for row in page_2_table[
    locality_start:
]:

    if len(row) < 7:
        continue


    disease_cell = clean(row[0])


    if disease_cell:
        current_disease = disease_cell


    if current_disease is None:
        continue


    raw_values = [
        clean(cell)
        for cell in row
        if clean(cell)
    ]


    raw_text = " | ".join(
        raw_values
    )


    district_cell = clean(row[4])


    possible_code = (
        district_cell
        .rstrip(":")
        .strip()
        .upper()
    )


    if possible_code in DISTRICTS:
        locality_text = clean(
            row[6]
        )

        if not locality_text:
            continue

        locality_records.append({
            "disease_raw": current_disease,
            "disease": normalize_disease(
                current_disease
            ),
            "district_code": possible_code,
            "district_reported_count":
                optional_count(row[5]),
            "locality_text": locality_text,
            "raw_text": raw_text,
        })

        continue


    district_sections = (
        split_district_sections(
            district_cell
        )
    )


    for district_code, locality_text in (
        district_sections
    ):
        locality_records.append({
            "disease_raw": current_disease,
            "disease": normalize_disease(
                current_disease
            ),
            "district_code": district_code,

            # The combined row does not give
            # district-specific counts.
            "district_reported_count":
                pd.NA,

            "locality_text": locality_text,
            "raw_text": raw_text,
        })

In [28]:
locality_df = pd.DataFrame(
    locality_records
)


if locality_df.empty:
    raise ValueError(
        "No locality records were extracted."
    )


locality_df[
    "district_reported_count"
] = locality_df[
    "district_reported_count"
].astype("Int64")


locality_df["district_name"] = (
    locality_df["district_code"]
    .map(DISTRICTS)
)


if locality_df[
    "district_name"
].isna().any():
    raise ValueError(
        "A district name could not be mapped."
    )


duplicate_localities = (
    locality_df.duplicated(
        subset=[
            "disease",
            "district_code",
        ],
        keep=False
    )
)


if duplicate_localities.any():
    duplicates = locality_df.loc[
        duplicate_localities,
        [
            "disease",
            "district_code",
            "locality_text",
        ]
    ]

    raise ValueError(
        "Duplicate disease-district sections found:\n"
        + duplicates.to_string(index=False)
    )


print(
    "Locality records:",
    len(locality_df)
)


locality_df[
    [
        "disease",
        "district_code",
        "district_name",
        "district_reported_count",
        "locality_text",
    ]
]

Locality records: 24


,disease,district_code,district_name,district_reported_count,locality_text
0,Dengue,TVM,Thiruvananthapuram,20,"Nemom, Malayinkeezhu, Attukal, Veli, Nanthanco..."
1,Dengue,KLM,Kollam,11,"Kilikolloor 1, KS Puram 2 ,Perinad 1 ,Pooyappa..."
2,Dengue,PTA,Pathanamthitta,3,"Ranni, Pandalam, Kulanada"
3,Dengue,IDK,Idukki,1,THODUPUZHA
4,Dengue,EKM,Ernakulam,7,"Thripunithura-3, Binanipuram-2, Kadungalloor, ..."
5,Dengue,TSR,Thrissur,22,"Avanur 1, Chalakudy Muncipality 2, Cherpu 1, C..."
6,Dengue,PKD,Palakkad,3,"Palakkad, Pudussery, Pirayiri"
7,Dengue,MPM,Malappuram,10,"Anakkayam, Porur , Thiruvally , Thevarkadapura..."
8,Dengue,WYD,Wayanad,1,Chethalayam
9,Dengue,KNR,Kannur,1,CHERUKUNNU


In [29]:
kannur_localities = locality_df[
    locality_df[
        "district_code"
    ] == "KNR"
]


kannur_localities[
    [
        "disease",
        "district_reported_count",
        "locality_text",
    ]
]

,disease,district_reported_count,locality_text
9,Dengue,1,CHERUKUNNU
17,Leptospirosis,<NA>,CHIRAKKAL


In [30]:
death_lines = []


for row in page_3_table:
    for cell in row:

        if not isinstance(
            cell,
            str
        ):
            continue

        if (
            "Death:" in cell
            and "DOD" in cell
        ):
            lines = cell.splitlines()

            for line in lines:
                line = clean(line)

                if line:
                    death_lines.append(
                        line
                    )


print(
    "Death-note lines found:",
    len(death_lines)
)


for line in death_lines:
    print(line)

Death-note lines found: 6
TVM: Dengue Death: 53/F, KUNNATHUKAL, DOD:-30.08.2026
EKM: Lepto Death: 56/M, Edavanakkad, DOD ;27.08.2026
KKD: Lepto Death: 74/M, VAYALADA, DOD 20.08.2026
MPM: Hep A Death: 25/M, Areacode, DOD 26.08.2026
PKD: H1N1 Death: 26/M, Agali, DOD : 28.08.2026
MPM: Amoebic Meningo Encephalitis Death: 38/M, Marakkara, DOD 31.08.2026


In [31]:
DEATH_PATTERN = re.compile(
    r"^"
    r"(?:(?P<district>[A-Z]{3}):\s*)?"
    r"(?P<disease>.+?)\s+Death:\s*"
    r"(?P<age>\d+)\s*/\s*"
    r"(?P<sex>[^,]+),\s*"
    r"(?P<locality>.+?),\s*"
    r"DOD\s*[:;]?\s*-?\s*"
    r"(?P<date>"
    r"\d{1,2}[./-]"
    r"\d{1,2}[./-]"
    r"\d{4}"
    r")"
    r"\s*$",
    re.IGNORECASE
)

In [32]:
SEX_ALIASES = {
    "M": "male",
    "MALE": "male",
    "F": "female",
    "FEMALE": "female",
}


death_records = []


for line in death_lines:
    match = DEATH_PATTERN.match(
        line
    )


    if match is None:
        death_records.append({
            "district_code": None,
            "district_name": None,
            "disease_raw": None,
            "disease": None,
            "age": pd.NA,
            "sex": None,
            "locality": None,
            "death_date": None,
            "parse_status": "needs_review",
            "raw_text": line,
        })

        continue


    district_code = match.group(
        "district"
    )


    if district_code:
        district_code = (
            district_code.upper()
        )


    disease_raw = clean(
        match.group("disease")
    )


    sex_raw = clean(
        match.group("sex")
    ).upper()


    date_text = match.group(
        "date"
    )


    date_text = re.sub(
        r"[-/]",
        ".",
        date_text
    )


    death_date = datetime.strptime(
        date_text,
        "%d.%m.%Y"
    ).date().isoformat()


    death_records.append({
        "district_code": district_code,

        "district_name":
            DISTRICTS.get(
                district_code
            ),

        "disease_raw": disease_raw,

        "disease": normalize_disease(
            disease_raw
        ),

        "age": int(
            match.group("age")
        ),

        "sex": SEX_ALIASES.get(
            sex_raw,
            sex_raw.casefold()
        ),

        "locality": clean(
            match.group("locality")
        ),

        "death_date": death_date,

        "parse_status": "parsed",

        "raw_text": line,
    })

In [33]:
DEATH_COLUMNS = [
    "district_code",
    "district_name",
    "disease_raw",
    "disease",
    "age",
    "sex",
    "locality",
    "death_date",
    "parse_status",
    "raw_text",
]


death_df = pd.DataFrame(
    death_records,
    columns=DEATH_COLUMNS
)


if not death_df.empty:
    death_df["age"] = (
        death_df["age"]
        .astype("Int64")
    )


    needs_review = death_df[
        death_df["parse_status"]
        == "needs_review"
    ]


    if not needs_review.empty:
        print(
            "WARNING: Some death notes "
            "need manual review."
        )

        print(
            needs_review[
                ["raw_text"]
            ].to_string(index=False)
        )


print(
    "Death records:",
    len(death_df)
)


death_df

Death records: 6


,district_code,district_name,disease_raw,disease,age,sex,locality,death_date,parse_status,raw_text
0,TVM,Thiruvananthapuram,Dengue,Dengue,53,female,KUNNATHUKAL,2026-08-30,parsed,"TVM: Dengue Death: 53/F, KUNNATHUKAL, DOD:-30...."
1,EKM,Ernakulam,Lepto,Leptospirosis,56,male,Edavanakkad,2026-08-27,parsed,"EKM: Lepto Death: 56/M, Edavanakkad, DOD ;27.0..."
2,KKD,Kozhikode,Lepto,Leptospirosis,74,male,VAYALADA,2026-08-20,parsed,"KKD: Lepto Death: 74/M, VAYALADA, DOD 20.08.2026"
3,MPM,Malappuram,Hep A,Hepatitis A,25,male,Areacode,2026-08-26,parsed,"MPM: Hep A Death: 25/M, Areacode, DOD 26.08.2026"
4,PKD,Palakkad,H1N1,Influenza A (H1N1),26,male,Agali,2026-08-28,parsed,"PKD: H1N1 Death: 26/M, Agali, DOD : 28.08.2026"
5,MPM,Malappuram,Amoebic Meningo Encephalitis,Amoebic Meningoencephalitis,38,male,Marakkara,2026-08-31,parsed,"MPM: Amoebic Meningo Encephalitis Death: 38/M,..."


In [34]:
locality_df.insert(
    0,
    "report_date",
    report_date
)

locality_df.insert(
    1,
    "period_type",
    "daily"
)

locality_df.insert(
    2,
    "source_filename",
    PDF_PATH.name
)

locality_df.insert(
    3,
    "schema_version",
    "locality_v1"
)

locality_df.insert(
    4,
    "source_page",
    2
)


death_df.insert(
    0,
    "report_date",
    report_date
)

death_df.insert(
    1,
    "period_type",
    "daily"
)

death_df.insert(
    2,
    "source_filename",
    PDF_PATH.name
)

death_df.insert(
    3,
    "schema_version",
    "death_notes_v1"
)

death_df.insert(
    4,
    "source_page",
    3
)

In [35]:
locality_output_path = (
    OUTPUT_FOLDER
    / f"localities_{report_date}.csv"
)


death_output_path = (
    OUTPUT_FOLDER
    / f"death_notes_{report_date}.csv"
)


locality_df.to_csv(
    locality_output_path,
    index=False
)


death_df.to_csv(
    death_output_path,
    index=False
)


print("Locality data saved:")
print(locality_output_path)

print("\nDeath-note data saved:")
print(death_output_path)

Locality data saved:
C:\Users\vinee\Downloads\rag_chatbot_kerala\data\processed\localities_2026-09-01.csv

Death-note data saved:
C:\Users\vinee\Downloads\rag_chatbot_kerala\data\processed\death_notes_2026-09-01.csv


In [36]:
parsed_deaths = 0
review_deaths = 0


if not death_df.empty:
    parsed_deaths = int(
        death_df[
            "parse_status"
        ].eq("parsed").sum()
    )

    review_deaths = int(
        death_df[
            "parse_status"
        ].eq("needs_review").sum()
    )


print("Notebook 03 completed successfully.")

print("Report date:", report_date)

print(
    "Locality records:",
    len(locality_df)
)

print(
    "Death records parsed:",
    parsed_deaths
)

print(
    "Death records needing review:",
    review_deaths
)

Notebook 03 completed successfully.
Report date: 2026-09-01
Locality records: 24
Death records parsed: 6
Death records needing review: 0
